# 36 — Angle-metric (v3) figures for experiments that logged but never plotted them

Offline notebook: no model, no GPU — pure pandas/matplotlib over existing
JSONL logs. Three figures:

1. **Beam singles in cosine units** (notebook 31's log): the greedy-vs-beam
   result re-expressed in the bounded token-cosine track. The reused
   greedy-λ0 cell (notebook 24) predates metric v3 and is absent by
   construction.
2. **Decomposition triptych in joint_cos** (notebooks 33 + 34): does the
   "common-only ≈ full blend, difference-only anti-composes" localization
   hold in scale-free units?
3. **bf16 pool steerability in d_cos** (notebook 32's validation): the
   validated concepts' scale-free single-concept steerability.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# --- Figure 1: notebook 31 singles, cosine track ---
rows = []
for line in Path("resources/31_single_beam.jsonl").read_text().strip().split("\n"):
    r = json.loads(line)
    c = r["config"]
    rows.append(
        {
            "search": c["search"],
            "lam": c["lam"],
            "concept": c["concept"],
            "d_cos": r["angles"][c["concept"]]["delta"]["token_cos"],
        }
    )
singles = pd.DataFrame(rows)
cells = [("greedy", 0.4), ("beam", 0.0), ("beam", 0.4)]
summary31 = (
    singles.groupby(["search", "lam"])["d_cos"]
    .agg(["mean", lambda s: (s > 0).mean()])
    .rename(columns={"mean": "d_cos", "<lambda_0>": "positive"})
    .reindex(cells)
)
print(summary31.round(4).to_string())

per_concept = (
    singles.groupby(["concept", "search", "lam"])["d_cos"].mean().unstack([1, 2])
)
g04, b0 = per_concept[("greedy", 0.4)], per_concept[("beam", 0.0)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
labels = [f"{s}\nlam={lam}" for s, lam in cells]
axes[0].bar(
    range(len(cells)),
    summary31["d_cos"],
    color=["tab:cyan", "tab:blue", "tab:purple"],
)
axes[0].set_xticks(range(len(cells)), labels, fontsize=9)
axes[0].set_ylabel("mean token-cosine delta (bounded, scale-free)")
axes[0].axhline(0, color="gray", lw=0.8)
axes[0].set_title("single-concept steering, angle track\n(greedy lam=0 predates v3)")

axes[1].scatter(g04, b0, s=18)
lim = [min(g04.min(), b0.min()) - 0.005, max(g04.max(), b0.max()) + 0.005]
axes[1].plot(lim, lim, color="gray", lw=0.8, ls="--")
axes[1].set_xlabel("greedy lam=0.4 d_cos (per concept)")
axes[1].set_ylabel("beam lam=0 d_cos (per concept)")
axes[1].set_title(
    f"beam wins on {(b0 > g04).sum()}/{len(per_concept)} concepts (cosine track)"
)
plt.tight_layout()
plt.savefig("resources/36_singles_cos.png", dpi=150)
plt.show()

In [ ]:
# --- Figure 2: decomposition triptych, joint_cos track (33 + 34) ---
def joint_cos_rows(path, methods):
    rows = []
    for line in Path(path).read_text().strip().split("\n"):
        r = json.loads(line)
        c = r["config"]
        if c["method"] not in methods:
            continue
        synsets = [s for s in (c["a"], c["b"], c.get("c")) if s]
        rows.append(
            {
                "method": c["method"],
                "n": c["n"],
                "group": " + ".join(synsets),
                "joint_cos": min(r["angles"][s]["delta"]["token_cos"] for s in synsets),
            }
        )
    return pd.DataFrame(rows)


fresh = joint_cos_rows("resources/34_common_blend.jsonl", {"common-blend"})
reused = joint_cos_rows("resources/33_scaling3_bf16.jsonl", {"blend", "gds-blend"})
reused = reused[reused["group"].isin(set(fresh["group"]))]
trip = pd.concat([fresh, reused], ignore_index=True)
summary34 = (
    trip.groupby(["method", "n"])["joint_cos"]
    .agg(["mean", lambda s: (s > 0).mean(), "size"])
    .rename(columns={"mean": "joint_cos", "<lambda_0>": "all_positive_cos"})
)
print(summary34.round(4).to_string())

order = ["gds-blend", "blend", "common-blend"]
colors = {"gds-blend": "navy", "blend": "tab:blue", "common-blend": "tab:red"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, (col, label) in zip(
    axes,
    (
        ("joint_cos", "mean joint token-cosine gain"),
        ("all_positive_cos", "share with ALL concepts improved (cosine)"),
    ),
):
    for j, method in enumerate(order):
        vals = [summary34.loc[(method, n), col] for n in (2, 3)]
        ax.bar(
            [i + (j - 1) * 0.25 for i in range(2)],
            vals,
            width=0.24,
            color=colors[method],
            label=method,
        )
    ax.set_xticks(range(2), ["n=2", "n=3"])
    ax.set_ylabel(label)
    ax.axhline(0, color="gray", lw=0.8)
axes[0].legend(fontsize=8)
axes[1].axhline(0.5, color="gray", ls="--", lw=0.8)
plt.suptitle("decomposition triptych in the bounded angle track")
plt.tight_layout()
plt.savefig("resources/36_triptych_cos.png", dpi=150)
plt.show()

In [ ]:
# --- Figure 3: bf16 pool steerability, d_cos track (32) ---
pool = pd.read_csv("resources/32_pool_summary_bf16.csv")
validated = pool.dropna(subset=["d_cos"]).sort_values("d_cos", ascending=False)
print(
    f"{len(validated)}/{len(pool)} pool members have d_cos "
    "(incumbents predate metric v3)"
)

fig, ax = plt.subplots(figsize=(13, 4.6))
ax.bar(range(len(validated)), validated["d_cos"], color="tab:green")
ax.set_xticks(range(len(validated)))
ax.set_xticklabels(
    [s.split(".")[0] for s in validated["synset"]],
    rotation=90,
    fontsize=7,
)
ax.set_ylabel("validated single-concept d_cos")
ax.axhline(0, color="gray", lw=0.8)
ax.set_title("bf16 concept pool: scale-free steerability (token-cosine delta)")
plt.tight_layout()
plt.savefig("resources/36_pool_cos.png", dpi=150)
plt.show()

# cross-check: are the two tracks ranking concepts consistently?
both = pool.dropna(subset=["d_cos"])
print(
    "delta vs d_cos rank correlation (spearman):",
    round(both["delta"].corr(both["d_cos"], method="spearman"), 3),
)